In [ ]:
"""
File: Gaussian_Hidden_Markov_Mode_Segmentation.ipynb

Purpose:
    Performs regime segmentation on standardized factor data using a Gaussian Hidden Markov Model (HMM).
    Selects the number of regimes based on silhouette scores with BIC as a tie-breaker, then assigns
    regime labels for downstream regime-aware modeling and explainability.

Functions:
    bic_gaussian_hmm:
        Computes the Bayesian Information Criterion (BIC) for a fitted Gaussian HMM.

Workflow:
    - Fits Gaussian HMMs over a candidate range of regime counts.
    - Evaluates each model using BIC and silhouette metrics.
    - Selects an optimal, parsimonious number of regimes.
    - Fits a final HMM and assigns regime labels to the dataset.
"""

!pip -q install hmmlearn


import pandas as pd
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
from torch.utils.data import TensorDataset, DataLoader


if torch.backends.mps.is_available():
    device = torch.device("mps")
#else: (only using cpu for now)
device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/biweekly_smoothed.csv", index_col=0)
print("df shape:", df.shape)
print(df.head())


df shape: (1623, 27)
               Lo 10     2-Dec     3-Dec     4-Dec     5-Dec     6-Dec  \
date                                                                     
1963-07-07 -0.142292 -0.027292 -0.106667 -0.085833 -0.072083  0.070417   
1963-07-21  0.048083  0.115346 -0.018143  0.046350 -0.018478  0.013523   
1963-08-04 -0.077000 -0.112500 -0.171357 -0.125429 -0.136571 -0.207929   
1963-08-18  0.035786  0.041214  0.073143  0.052714  0.209143  0.100786   
1963-09-01  0.112571  0.126143  0.203714  0.228286  0.248214  0.199857   

               7-Dec     8-Dec     9-Dec     Hi 10  ...   RMW_res   CMA_res  \
date                                                ...                       
1963-07-07 -0.022083  0.027083 -0.072083  0.011042  ... -0.009449 -0.059750   
1963-07-21  0.027978  0.052856  0.031336  0.042845  ...  0.000946 -0.085031   
1963-08-04 -0.145286 -0.182071 -0.090500 -0.075500  ...  0.004952 -0.071465   
1963-08-18  0.142214  0.146429  0.228000  0.249571  ...  0.041113

In [ ]:
df = df.replace(-99.99, np.nan)
df = df.dropna()
df

,Lo 10,2-Dec,3-Dec,4-Dec,5-Dec,6-Dec,7-Dec,8-Dec,9-Dec,Hi 10,...,RMW_res,CMA_res,SMB_res,HML_nutr,RMW_nutr,CMA_nutr,SMB_nutr,SMB_constructed,SMB2,SMB_constructed_nutr
date,,,,,,,,,,,,,,,,,,,,,
1963-07-07,-0.142292,-0.027292,-0.106667,-0.085833,-0.072083,0.070417,-0.022083,0.027083,-0.072083,0.011042,...,-0.009449,-0.059750,-0.097602,-0.269362,-0.024003,-0.166747,-0.177467,-0.055600,-0.089708,-0.032225
1963-07-21,0.048083,0.115346,-0.018143,0.046350,-0.018478,0.013523,0.027978,0.052856,0.031336,0.042845,...,0.000946,-0.085031,-0.013105,-0.163311,0.002404,-0.237300,-0.023829,-0.032397,0.000924,-0.006389
1963-08-04,-0.077000,-0.112500,-0.171357,-0.125429,-0.136571,-0.207929,-0.145286,-0.182071,-0.090500,-0.075500,...,0.004952,-0.071465,-0.019858,-0.091544,0.012579,-0.199441,-0.036107,-0.028790,0.015686,-0.016206
1963-08-18,0.035786,0.041214,0.073143,0.052714,0.209143,0.100786,0.142214,0.146429,0.228000,0.249571,...,0.041113,-0.014553,-0.111994,0.092831,0.104439,-0.040614,-0.203637,-0.028478,-0.091000,-0.026303
1963-09-01,0.112571,0.126143,0.203714,0.228286,0.248214,0.199857,0.250500,0.225571,0.261643,0.212214,...,-0.006316,-0.003975,-0.034374,0.136683,-0.016045,-0.011094,-0.062501,-0.014888,-0.046171,-0.008854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-13,0.468095,0.515079,0.401587,0.468095,0.365794,0.357540,0.321746,0.311429,0.274286,0.285873,...,-0.080267,0.198244,0.103997,0.251864,-0.203903,0.553250,0.189095,0.073991,0.133556,0.043483
2025-07-27,0.439500,0.330429,0.249286,0.216571,0.224643,0.179429,0.171500,0.189214,0.111071,0.176643,...,-0.094825,0.014730,0.081997,-0.000871,-0.240886,0.041109,0.149094,0.000634,0.126514,0.019467
2025-08-10,-0.117214,-0.170357,-0.049500,-0.134714,-0.090214,-0.040500,-0.040714,0.030643,0.081143,0.096929,...,-0.004774,-0.155715,-0.157847,-0.321570,-0.012127,-0.434562,-0.287009,-0.106112,-0.137900,-0.054699


In [ ]:
factor_cols = ['Mkt-RF', 'SMB_constructed', 'HML', 'RMW', 'CMA', 'SMB2']
df_factors = df[factor_cols].copy()
df_factors

,Mkt-RF,SMB_constructed,HML,RMW,CMA,SMB2
date,,,,,,
1963-07-07,-0.018125,-0.055600,-0.136667,0.007708,-0.041667,-0.089708
1963-07-21,0.027188,-0.032397,-0.079812,0.014335,-0.072784,0.000924
1963-08-04,-0.108714,-0.028790,-0.025357,0.029643,-0.041714,0.015686
1963-08-18,0.206714,-0.028478,0.050786,0.039571,-0.025429,-0.091000
1963-09-01,0.208429,-0.014888,0.075929,-0.008000,-0.015071,-0.046171
...,...,...,...,...,...,...
2025-07-13,0.275635,0.073991,0.135952,-0.087540,0.178492,0.133556
2025-07-27,0.156786,0.000634,0.001500,-0.092214,0.010286,0.126514
2025-08-10,0.062929,-0.106112,-0.174571,0.005643,-0.148071,-0.137900


In [ ]:
# Standardize so each factor has mean=0, std=1
scaler = StandardScaler()
factors_scaled = scaler.fit_transform(df_factors)

Regime Segmentation


In [ ]:
import numpy as np
import warnings

# Ensure hmmlearn is available
try:
    from hmmlearn import hmm
except ImportError:
    # Colab / notebook install fallback
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "hmmlearn"])
    from hmmlearn import hmm

from sklearn.metrics import silhouette_score

X_hmm = factors_scaled  # standardized factor matrix [T, d]
n_samples, n_features = X_hmm.shape

def bic_gaussian_hmm(model, X, cov_type="full"):
    """Compute BIC for GaussianHMM fitted on X.
    BIC = -2*logL + p*log(n)
    where p is the number of free parameters.
    """
    logL = model.score(X)
    k = model.n_components
    d = X.shape[1]

    # free parameters:
    # startprob: (k-1)
    # transmat: k*(k-1)
    # means: k*d
    # covars:
    if cov_type == "full":
        cov_params = k * (d * (d + 1) / 2)
    elif cov_type == "diag":
        cov_params = k * d
    else:
        # fallback (treat as diag)
        cov_params = k * d

    p = (k - 1) + (k * (k - 1)) + (k * d) + cov_params
    return -2.0 * logL + p * np.log(X.shape[0])

# Candidate k values
k_min, k_max = 2, 6
k_max = min(k_max, max(k_min, int(np.sqrt(n_samples))))  # keep sensible upper bound
k_range = range(k_min, k_max + 1)

bic_scores = []
silhouette_scores = []

warnings.filterwarnings("ignore", category=RuntimeWarning)

for k in k_range:
    try:
        model = hmm.GaussianHMM(
            n_components=k,
            covariance_type="full",
            n_iter=200,
            random_state=42,
            tol=1e-3
        )
        model.fit(X_hmm)

        # BIC (lower is better)
        bic = bic_gaussian_hmm(model, X_hmm, cov_type="full")
        bic_scores.append(bic)

        # Silhouette (higher is better)
        labels = model.predict(X_hmm)
        if len(np.unique(labels)) < 2 or len(np.unique(labels)) >= n_samples:
            sil = np.nan
        else:
            sil = silhouette_score(X_hmm, labels)
        silhouette_scores.append(sil)

        print(f"Tested k={k}: BIC={bic:.2f}, Silhouette={sil:.4f}")
    except Exception as e:
        bic_scores.append(np.nan)
        silhouette_scores.append(np.nan)
        print(f"Tested k={k}: failed ({type(e).__name__}: {e})")

# Pick k: Silhouette recommended (tie-break by BIC)
valid_sil = np.array(silhouette_scores, dtype=float)
valid_bic = np.array(bic_scores, dtype=float)

if np.all(np.isnan(valid_sil)):
    # fallback to BIC only
    best_k = list(k_range)[int(np.nanargmin(valid_bic))]
    pick_reason = "BIC (fallback)"
else:
    best_sil = np.nanmax(valid_sil)
    cand = [list(k_range)[i] for i, v in enumerate(valid_sil) if np.isfinite(v) and abs(v - best_sil) < 1e-12]
    if len(cand) == 1:
        best_k = cand[0]
    else:
        # tie-break: choose lowest BIC among candidates
        cand_idx = [list(k_range).index(k) for k in cand]
        best_k = cand[int(np.nanargmin(valid_bic[cand_idx]))]
    pick_reason = "Silhouette (tie-break by BIC)"

n_regimes = best_k
print(f"\nSelected n_regimes = {n_regimes} via {pick_reason}")

# Fit final model and assign regime labels
final_model = hmm.GaussianHMM(
    n_components=n_regimes,
    covariance_type="full",
    n_iter=500,
    random_state=42,
    tol=1e-3
)
final_model.fit(X_hmm)
regime_labels = final_model.predict(X_hmm)

df["Regime"] = regime_labels

# quick sanity check
counts = df["Regime"].value_counts().sort_index()
print("\nRegime counts:")
print(counts)



Tested k=2: BIC=19531.68, Silhouette=0.2430
Tested k=3: BIC=19753.14, Silhouette=-0.0674
Tested k=4: BIC=19263.80, Silhouette=0.0247
Tested k=5: BIC=19558.47, Silhouette=-0.1136
Tested k=6: BIC=19459.73, Silhouette=-0.0465

Selected n_regimes = 2 via Silhouette (tie-break by BIC)

Regime counts:
Regime
0    1189
1     434
Name: count, dtype: int64
